# mutable_default_arguments
**Prerequisites:** memory_model, references, mutability

**Outcome:** After this notebook you will understand one of the most infamous Python gotchas, exactly why it happens at the memory level, and the two correct patterns to avoid it permanently.

## The Problem

In [ ]:
def add_job(job, queue=[]):
    queue.append(job)
    return queue

print(add_job("job_1"))   # expected: ["job_1"]
print(add_job("job_2"))   # expected: ["job_2"]
print(add_job("job_3"))   # expected: ["job_3"]
# why does each call accumulate previous results?

## The Concept

Default argument values in Python are evaluated exactly once — at function definition time, not at call time. The default object is created when the `def` statement executes and stored inside the function object itself. Every call that uses the default gets the same object. If that object is mutable — a list, dict, or set — mutations made during one call are visible in every subsequent call that uses the default. This is not a bug in Python. It is a direct consequence of how the memory model works. The surprise comes from assuming defaults are re-created on each call.

## Minimal Example

In [ ]:
def add_job(job, queue=[]):
    queue.append(job)
    return queue

# the default list lives inside the function object
print(add_job.__defaults__)    # ([],) at definition time
add_job("job_1")
print(add_job.__defaults__)    # (["job_1"],) — the default was mutated
add_job("job_2")
print(add_job.__defaults__)    # (["job_1", "job_2"],)

## How Python Does It

When Python compiles a function definition it evaluates all default expressions and stores the results in a tuple at `function.__defaults__`. This tuple is created once and lives for the entire lifetime of the function object. Each call that omits a parameter receives a reference to the object stored in that tuple — the same object every time. Immutable defaults like `None`, `0`, or `"hello"` are safe because they cannot be changed. Mutable defaults are dangerous because they can.

In [ ]:
def process(data, config={}, tags=[]):
    pass

# inspect the default objects
print(process.__defaults__)          # ({}, []) — created once
print(id(process.__defaults__[0]))   # id of the default dict
print(id(process.__defaults__[1]))   # id of the default list

# call the function without arguments
process.__defaults__[1].append("tag_from_outside")
print(process.__defaults__)          # the default list is now polluted

## Building Up

In [ ]:
# the classic symptom: state bleeds between calls
def record_event(event, log=[]):
    log.append(event)
    return log

print(record_event("boot"))        # ["boot"]
print(record_event("connected"))   # ["boot", "connected"] — not what you want
print(record_event("shutdown"))    # ["boot", "connected", "shutdown"]

In [ ]:
# same problem with a dict default
def set_config(key, value, config={}):
    config[key] = value
    return config

print(set_config("host", "localhost"))   # {"host": "localhost"}
print(set_config("port", 5432))          # {"host": "localhost", "port": 5432}
# a brand new caller gets a pre-populated config — silent data leak

In [ ]:
# immutable defaults are completely safe
def greet(name, prefix="Hello"):
    return f"{prefix}, {name}"

print(greet("alice"))          # Hello, alice
print(greet("bob", "Hi"))      # Hi, bob
print(greet("carol"))          # Hello, carol — prefix unchanged

In [ ]:
# the intentional use: using mutable default as a cache (rare, explicit)
def fibonacci(n, _cache={}):
    if n in _cache:
        return _cache[n]
    if n <= 1:
        return n
    _cache[n] = fibonacci(n - 1, _cache) + fibonacci(n - 2, _cache)
    return _cache[n]

print(fibonacci(10))   # 55
print(fibonacci(30))   # 832040
# here the persistent default is intentional — it acts as a memo cache
# the leading underscore signals "this is an implementation detail, not a real argument"

## Where It Breaks

In [ ]:
# hard to debug: the bug only appears after multiple calls
def build_batch(job_id, batch=[]):
    batch.append(job_id)
    if len(batch) >= 3:
        result = batch[:]
        batch.clear()
        return result
    return None

print(build_batch("job_1"))   # None
print(build_batch("job_2"))   # None
print(build_batch("job_3"))   # ["job_1", "job_2", "job_3"] — works by accident
print(build_batch("job_4"))   # None — but the cleared list is still the default
print(build_batch("job_5"))   # None
print(build_batch("job_6"))   # ["job_4", "job_5", "job_6"]
# appears to work but relies entirely on the mutable default persisting

In [ ]:
# breaks completely in concurrent contexts
# two threads calling the same function share the same default list
# — neither thread gets a clean slate
import threading

def collect(item, store=[]):
    store.append(item)
    return store[:]

results = []
def task(name):
    results.append(collect(name))

threads = [threading.Thread(target=task, args=(f"worker_{i}",)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

print(results)   # each thread sees the accumulation from other threads

## The Fix

In [ ]:
# fix 1: use None as the sentinel, create a fresh object inside the function
def add_job(job, queue=None):
    if queue is None:
        queue = []          # new list created on every call that omits queue
    queue.append(job)
    return queue

print(add_job("job_1"))   # ["job_1"]
print(add_job("job_2"))   # ["job_2"] — fresh list every time
print(add_job("job_3"))   # ["job_3"]

In [ ]:
# fix 2: same pattern for dict defaults
def set_config(key, value, config=None):
    if config is None:
        config = {}
    config[key] = value
    return config

print(set_config("host", "localhost"))   # {"host": "localhost"}
print(set_config("port", 5432))          # {"port": 5432} — no bleed

## In a Real System

In [ ]:
# a pipeline stage that must never share state between invocations
def run_stage(records, errors=None, metadata=None):
    """
    Process a list of records.
    errors and metadata are isolated per call.
    """
    if errors is None:
        errors = []
    if metadata is None:
        metadata = {"processed": 0, "failed": 0}

    for record in records:
        if not isinstance(record.get("id"), str):
            errors.append(record)
            metadata["failed"] += 1
        else:
            metadata["processed"] += 1

    return {"errors": errors, "metadata": metadata}

batch_1 = [{"id": "job_1"}, {"id": None}, {"id": "job_3"}]
batch_2 = [{"id": "job_4"}, {"id": "job_5"}]

print(run_stage(batch_1))
print(run_stage(batch_2))   # completely independent result

## Performance

The `None` sentinel pattern adds one `is None` check per call — effectively zero overhead. The alternative of creating a new empty list or dict on every call is O(1) allocation. In practice the performance difference is unmeasurable. The intentional mutable default cache pattern (as shown with fibonacci) is a legitimate optimisation but should always be documented explicitly and never used in code that is called concurrently.

## Anti-Pattern

In [ ]:
# anti-pattern: mutable default argument in any production function
def tag_record(record, tags=[]):     # never do this
    tags.append("processed")
    record["tags"] = tags
    return record

r1 = tag_record({"id": "job_1"})
r2 = tag_record({"id": "job_2"})
print(r1)   # {"id": "job_1", "tags": ["processed", "processed"]} — wrong
print(r2)   # {"id": "job_2", "tags": ["processed", "processed"]} — same object!

# correct
def tag_record(record, tags=None):
    if tags is None:
        tags = []
    tags.append("processed")
    record["tags"] = tags
    return record

r1 = tag_record({"id": "job_1"})
r2 = tag_record({"id": "job_2"})
print(r1)   # {"id": "job_1", "tags": ["processed"]}
print(r2)   # {"id": "job_2", "tags": ["processed"]}

## Interview Signals

- When are default argument values evaluated in Python?
- Where does Python store default argument values and how can you inspect them?
- What is the correct pattern to use when you need a mutable default argument?
- Can you think of a case where a mutable default argument is intentional and useful?

## Exercise

In [ ]:
def create_pipeline(name, stages=[], config={}):
    """
    Creates a pipeline definition dict.
    Each call must return a completely independent pipeline.
    Bug: stages and config are shared across calls.
    Fix the function without changing its external interface.
    """
    stages.append("init")
    config["name"] = name
    return {"name": name, "stages": stages, "config": config}


p1 = create_pipeline("etl")
p2 = create_pipeline("streaming")

assert p1["name"] == "etl"
assert p2["name"] == "streaming"
assert p1["stages"] == ["init"],       f"got {p1['stages']}"
assert p2["stages"] == ["init"],       f"got {p2['stages']}"
assert p1["config"] == {"name": "etl"}, f"got {p1['config']}"
assert p1["stages"] is not p2["stages"], "stages must be independent objects"
assert p1["config"] is not p2["config"], "config must be independent objects"
print("all assertions passed")